## CIC IIoT Dataset 2025

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler



## Setting up paths
 define the paths for the combined data files (attack and benign)

In [ ]:

benign_path = r"C:\Users\NTC -\Downloads\iiot-dataset-2025\all\merged_benign_data.csv"
attack_path = r"C:\Users\NTC -\Downloads\iiot-dataset-2025\all\merged_attack_data.csv"

files = [attack_path, benign_path]
dfs = []


## Uploading Files

In [3]:
for file in files:
    print(f"Loading {file} ...")
    df_part = pd.read_csv(file)
    dfs.append(df_part)


Loading C:\Users\NTC -\Downloads\iiot-dataset-2025\all\merged_attack_data.csv ...
Loading C:\Users\NTC -\Downloads\iiot-dataset-2025\all\merged_benign_data.csv ...


## Data Merging and Overall Display

In [4]:
df = pd.concat(dfs, ignore_index=True)
print(f"\nCombined dataset shape: {df.shape[0]:,} rows, {df.shape[1]} columns")



Combined dataset shape: 685,671 rows, 94 columns


## Classification column detection and distribution display

In [6]:
label_col = None
for col in df.columns:
    if "label2" in col.lower():
        label_col = col
        break

if label_col:
    label_percent = (df[label_col].value_counts(normalize=True) * 100).round(2)
    print("\nLabel distribution (%):")
    print(label_percent)



Label distribution (%):
label2
benign        58.44
recon         15.44
dos            8.42
ddos           8.27
mitm           3.72
malware        3.53
web            1.32
bruteforce     0.88
Name: proportion, dtype: float64


## Handling Missing Values

In [7]:
missing_summary = df.isnull().sum()
total_missing = missing_summary.sum()
if total_missing > 0:
    df.fillna(df.median(numeric_only=True), inplace=True)
    print(f"\nMissing values filled: {int(total_missing):,}")
else:
    print("\nNo missing values detected.")



No missing values detected.


## Handling Outliers (IQR Winsorization)

In [8]:
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
rows_before = len(df)

for i, col in enumerate(numeric_cols, start=1):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    df[col] = np.where(df[col] < lower, lower, np.where(df[col] > upper, upper, df[col]))
    if i % 25 == 0 or i == len(numeric_cols):
        print(f"Processed {i}/{len(numeric_cols)} numeric columns for outliers...")


Processed 25/71 numeric columns for outliers...
Processed 50/71 numeric columns for outliers...
Processed 71/71 numeric columns for outliers...


## Normalization using RobustScaler

In [9]:
scaler = RobustScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])
print(f"\nNormalized {len(numeric_cols)} numeric columns.")



Normalized 71 numeric columns.


## Saving the purified data
save the output in a file `cleaned_merged_full.csv` .

In [ ]:
output_file = "cleaned_merged_full.csv"
df.to_csv(output_file, index=False)

